# 27. Hand-count validation month — January 2023

Validates whether the **healthy-regime** model counts are real. mushroom.pt reports its
highest abundance here (~9.8/scene); Feb-2024 hand-counting proved the *low* 2024 counts are
model recall failure (truth 9.7 vs model 0.52). If the model roughly matches your eye HERE in
Jan 2023 but badly under-counts in Feb 2024, the pre/post-Sep-2023 footage change is confirmed
as the cause and v2 retraining on the later footage is justified.

Same workflow as nb 26: set `IDX`, look, set `MY_COUNT`, Shift+Enter. No CSV editing.

In [1]:
import csv
from pathlib import Path

# Jan-2023 frames + blank CSV were built by scripts/setup_handcount_month.py 2023/01
FRAMES_DIR = Path('./handcount_2023_01_frames')
HANDCOUNT_CSV = Path('./handcount_2023_01.csv')
frames = sorted(FRAMES_DIR.glob('*.png'))
print(f'{len(frames)} frames ready; CSV = {HANDCOUNT_CSV}')

31 frames ready; CSV = handcount_2023_01.csv


In [36]:
# ── Hand-count viewer + recorder — ONE frame at a time ──
#   1. Set IDX to the frame you want. Leave MY_COUNT = None to just LOOK.
#   2. Count the worms, set MY_COUNT = <your number>, Shift+Enter: it saves to the CSV.
#   3. Bump IDX, repeat. Re-running a frame with a new number overwrites it (fix mistakes freely).
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

IDX = 31            # <-- which frame: 0 .. 30
MY_COUNT = 12    # <-- None = view only;  a number = record THIS frame's worm count

f = frames[IDX]
stem = f.stem
fig, ax = plt.subplots(figsize=(15, 8))
ax.imshow(mpimg.imread(f))
ax.set_axis_off()
ax.set_title(f'[{IDX + 1}/{len(frames)}]  {stem}', fontsize=12)
plt.show()

with HANDCOUNT_CSV.open() as fh:
    reader = csv.DictReader(fh)
    fields = list(reader.fieldnames)
    rows = list(reader)

if MY_COUNT is not None:
    matched = False
    for r in rows:
        if r['stem'] == stem:
            r['human_count'] = MY_COUNT
            matched = True
    if matched:
        with HANDCOUNT_CSV.open('w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=fields)
            w.writeheader()
            w.writerows(rows)
        print(f'saved  human_count = {MY_COUNT}  for {stem}')
    else:
        print(f'!! no CSV row matches {stem} — nothing written')

filled = sum(1 for r in rows if str(r.get('human_count', '')).strip())
print(f'progress: {filled}/{len(frames)} frames counted')
if MY_COUNT is not None and IDX + 1 < len(frames):
    print(f'-> next: set IDX = {IDX + 1}, MY_COUNT = <count for that frame>')

IndexError: list index out of range